# DD-PRiSM-plus — Step 1: set up and fetch all data

Run once in a **CPU** session. Click **Save Version** at the end or everything is lost.

**Session options (right-hand `<` panel):** Accelerator → **None**, Internet → **On**

> **figshare is down site-wide** — it answers `202 Accepted` with an empty body,
> `figshare.com` included. Three DepMap files come from there, so supply them
> yourself from the [DepMap 23Q4 downloads page](https://depmap.org/portal/data_page/?tab=allData):
>
> | file | size |
> |---|---|
> | `OmicsExpressionProteinCodingGenesTPMLogp1.csv` | 449.8 MB |
> | `Model.csv` | 0.5 MB |
>
> Upload them as a Kaggle Dataset and attach with **Add Input → Datasets**.
> Step 4 finds them wherever they land.

In [ ]:
# 0. Session check
import subprocess, os, glob, shutil

ok = subprocess.run(['curl', '-sI', '--max-time', '15', 'https://github.com'],
                    capture_output=True).returncode == 0
print('internet:', 'ON' if ok else 'OFF  <-- enable it, then rerun')
print('free disk:', subprocess.run(['df', '-h', '/kaggle/working'],
      capture_output=True, text=True).stdout.splitlines()[-1])

## 1. Get the code

In [ ]:
REPO = '/kaggle/working/ddprism-plus'
DATA = '/kaggle/working/data'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone --quiet https://github.com/SanaNiroomand/DD-PRiSM-plus.git {REPO}

os.chdir(REPO)
print('working in', os.getcwd())

## 2. Install what Kaggle lacks

`zipfile-deflate64` is **mandatory** — DOSERESP.zip is Deflate64 and the
standard library cannot decompress it.

In [ ]:
!pip install --quiet zipfile-deflate64 rdkit openpyxl
print('installed')

## 3. Check the model code

In [ ]:
!python -m pytest tests -q

## 4. Copy in the DepMap files you supplied

Accepts either DepMap naming. Harmless if you have not attached a dataset.

In [ ]:
os.makedirs(DATA, exist_ok=True)

# what you might have uploaded  ->  what the pipeline expects
ALIASES = {
    'OmicsExpressionProteinCodingGenesTPMLogp1.csv':      'OmicsExpressionProteinCodingGenesTPMLogp1.csv',
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv':  'OmicsExpressionProteinCodingGenesTPMLogp1.csv',
    'Model.csv':            'Model.csv',
    'sample_info.csv':      'sample_info_18q3.csv',
    'sample_info_18q3.csv': 'sample_info_18q3.csv',
}

for found, wanted in ALIASES.items():
    target = os.path.join(DATA, wanted)
    if os.path.exists(target):
        continue
    hits = glob.glob('/kaggle/input/**/' + found, recursive=True)
    if hits:
        shutil.copy(hits[0], target)
        print('copied', wanted, ' <- ', hits[0])
        if found != wanted:
            print('  NOTE:', found, 'is a post-23Q4 release. It works, but the')
            print('  paper row counts and metrics were produced with 23Q4.')

print()
print('contents of', DATA)
!ls -la {DATA} 2>/dev/null || echo '  (empty)'

## 5. Download everything else (~1 GB)

Files land **directly in `DATA`**. Already-present files are skipped, so this
is safe to rerun.

In [ ]:
!python scripts/get_data.py --dest {DATA} --attempts 3

## 6. Retry stragglers

Only useful if figshare has recovered. Skip it if step 4 already supplied both.

In [ ]:
!python scripts/get_data.py --dest {DATA} --only depmap_expression depmap_samples --attempts 3

## 7. Verify — the cell that matters

Every required row must read `ok`. Size, archive integrity and (for the DepMap
files) the official MD5 are all checked.

In [ ]:
!python scripts/get_data.py --dest {DATA} --check
print()
!du -sh {DATA}

## 8. Save it

**Save Version → Save & Run All (Commit).** Without this everything here is
deleted when the session ends.

The next notebook attaches this via **Add Input → Your Work → Notebook Output**.

---

**Next:** preprocessing. Success is exactly **7,915,900** NCI60 training rows
and **1,387,317** combination rows.